# 5. From model to service: packaging and the API

A model on your laptop helps nobody. This notebook covers the first two steps of the talk:

1. **Package** it, so it can be reused
2. **Wrap it in an API**, so other systems can call it

In [1]:
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["IMBUTO_ROOT"] = str(ROOT)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 10})
GREEN, GREY, BROWN = "#1f6e3f", "#9aa89f", "#8c510a"

## Step 1: package a bundle

A pickle file on its own raises questions. Which version? What data? What are the limits? Has it been changed? A **bundle** answers them: weights, a model card and checksums, in one folder.

In [2]:
bundle_dir = ROOT / "artifacts/crop_health"
for f in sorted(bundle_dir.iterdir()): print(f"{f.name:16}{f.stat().st_size / 1e6:6.2f} MB")
manifest = json.loads((bundle_dir / "bundle.json").read_text())
print(json.dumps({k: manifest["card"][k] for k in ("task", "training_data", "intended_use", "not_for", "known_limits")}, indent=1))

bundle.json       0.00 MB
centroid.json     0.01 MB
model.onnx        9.98 MB
{
 "task": "Bean leaf diagnosis from a phone photo",
 "training_data": "iBean (AI-Lab-Makerere/beans, MIT): field photos from Uganda",
 "intended_use": "Decision support for bean leaves, confirmed by an agronomist",
 "not_for": "Other crops, pesticide selection, or use without human follow-up",
 "known_limits": [
  "No Burundian test images yet: accuracy here may not hold in Burundi",
  "Beans only, and only two diseases",
  "Leaf check tested on small everyday images, not other crops: other leaves may pass"
 ]
}


The checksum makes loading safe. If a file is corrupted during a slow transfer, the service refuses to start, rather than giving wrong answers in the field.

In [3]:
import shutil, tempfile
from imbuto.ml.bundle import load_bundle, BundleError
with tempfile.TemporaryDirectory() as tmp:
    copy = Path(tmp) / "bundle"; shutil.copytree(bundle_dir, copy)
    print("Clean copy loads:", load_bundle(copy)["version"])
    (copy / "centroid.json").write_text("[0]")
    try: load_bundle(copy)
    except BundleError as error: print("Altered copy refused:", error)

Clean copy loads: 1.0.0
Altered copy refused: crop_health: centroid.json is missing or has been altered


**ONNX** frees us from the training framework. The serving install needs no PyTorch or TensorFlow, and no scikit-learn. Fewer dependencies mean a smaller image, faster starts and less to break.

## Step 2: load once, predict many

Loading takes time, so do it once at startup. Predicting is then quick.

In [4]:
import io, time
import pandas as pd
from imbuto.ml.crop_health import CropHealthModel, shrink_for_upload
from PIL import Image
start = time.perf_counter(); model = CropHealthModel(bundle_dir); print(f"Load: {(time.perf_counter() - start) * 1000:.0f} ms")
test = pd.read_parquet(ROOT / "data/bronze/beans/test.parquet")
photo = shrink_for_upload(Image.open(io.BytesIO(test.image.iloc[50]["bytes"])))
start = time.perf_counter(); result = model.predict(photo); print(f"Predict: {(time.perf_counter() - start) * 1000:.0f} ms"); print(result)

Load: 88 ms

Predict: 8 ms
Diagnosis(label='angular_leaf_spot', confidence=0.762, probabilities={'angular_leaf_spot': 0.762, 'bean_rust': 0.238, 'healthy': 0.0}, similarity=0.79, outcome='confident')


## Step 3: the API

FastAPI gives us validation and interactive docs at `/api/docs` for free. Here is the idea in its smallest form:

```python
app = FastAPI()
model = CropHealthModel(Path("artifacts/crop_health"))  # once, at startup

@app.post("/api/v1/diagnose")
def diagnose(image: UploadFile):                         # `def`, not `async def`:
    return model.predict(image.file.read())              # inference is CPU work, so FastAPI
                                                         # runs it in a thread pool
```

The real service, in `src/imbuto/api/main.py`, adds what production needs. We call it below through a test client, exactly as another system would over HTTP.

In [5]:
import tempfile
from fastapi.testclient import TestClient
from imbuto.api.main import create_app
from imbuto.core.service import Imbuto
from imbuto.settings import Settings
client = TestClient(create_app(Imbuto(Settings(db_path=Path(tempfile.mkdtemp()) / "demo.sqlite"))))
print(client.get("/readyz").json())
response = client.post("/api/v1/diagnose", files={"image": ("leaf.jpg", photo, "image/jpeg")}, data={"lang": "en"})
print(response.status_code, response.json())

/Users/desmondonnam/Desktop/projects/AI/Imbuto_AI/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'status': 'ready', 'version': '0.1.0', 'models': {'crop_health': '1.0.0', 'rain_data_through': '2026-09'}}
200 {'prediction_id': 1, 'message': 'Likely: Angular leaf spot. Confirm with an adviser.', 'model_version': '1.0.0', 'label': 'angular_leaf_spot', 'confidence': 0.762, 'probabilities': {'angular_leaf_spot': 0.762, 'bean_rust': 0.238, 'healthy': 0.0}, 'similarity': 0.79, 'outcome': 'confident'}


### What production adds

A demo handles the happy path. A service handles everything else.

In [6]:
cases = {
    "not an image": client.post("/api/v1/diagnose", files={"image": ("a.jpg", b"hello", "image/jpeg")}).status_code,
    "too large": client.post("/api/v1/diagnose", files={"image": ("a.jpg", b"x" * 2_000_001, "image/jpeg")}).status_code,
    "bad language": client.post("/api/v1/advice", json={"question": "hello", "lang": "xx"}).status_code,
    "unknown commune": client.get("/api/v1/rain/NOPE").status_code,
}
pd.Series(cases).rename("HTTP status").to_frame()

,HTTP status
not an image,422
too large,413
bad language,422
unknown commune,404


## The advisory assistant: RAG without the risk

Retrieval-augmented generation has two halves. Today we build retrieval only, and quote approved text. Nothing unvetted can reach a farmer, and it runs on any small server.

Kirundi words change with prefixes and suffixes, and there is no standard tokeniser for it. So we match overlapping **character 4-grams**, which needs no language tools at all.

In [7]:
from imbuto.ml.advisory import _grams
print(_grams("ibiharage")); print(_grams("igiharage"))
print("Shared 4-grams:", set(_grams("ibiharage")) & set(_grams("igiharage")))

[' ibi', 'ibih', 'biha', 'ihar', 'hara', 'arag', 'rage', 'age ']
[' igi', 'igih', 'giha', 'ihar', 'hara', 'arag', 'rage', 'age ']
Shared 4-grams: {'rage', 'ihar', 'age ', 'arag', 'hara'}


In [8]:
questions = ["How do I store my maize after harvest?", "mes haricots ont des taches orange", "Ibiharage vyanje bifise indwara ku mababi",
             "what pesticide dose for tomatoes", "what is the capital of France"]
rows = []
for q in questions:
    body = client.post("/api/v1/advice", json={"question": q, "lang": "en"}).json()
    rows.append({"question": q, "score": body["score"], "source": body["source"], "handed to a person": body["escalated"]})
pd.DataFrame(rows)

,question,score,source,handed to a person
0,How do I store my maize after harvest?,0.422,storage-drying,False
1,mes haricots ont des taches orange,0.667,bean-rust,False
2,Ibiharage vyanje bifise indwara ku mababi,0.557,angular-leaf-spot,False
3,what pesticide dose for tomatoes,0.083,NaN,True
4,what is the capital of France,0.129,NaN,True


Note that the pesticide question goes to a person. That is the behaviour we want.

An LLM could later rewrite retrieved passages into a friendlier form, as a second step behind the same safety checks. **All knowledge-base content here is an unreviewed draft** until an agronomist approves it.

**Next:** notebook 6 puts it in front of users, in a container, and watches it.